In [2]:
import pandas as pd 
import numpy as np 

pd.set_option('display.max_columns',None)

In [3]:
sp = pd.read_csv('data/raw/sales_pipeline.csv')
acc= pd.read_csv('data/raw/accounts.csv')
prod = pd.read_csv('data/raw/products.csv')
teams = pd.read_csv('data/raw/sales_teams.csv')
leads = pd.read_csv('data/raw/lead_sources.csv')

In [4]:
for name, df in [('sales_pipeline', sp), ('accounts', acc), ('products', prod),
                  ('sales_teams', teams), ('lead_sources', leads)]:
    print(f"{name}: {df.shape}")

sales_pipeline: (8800, 8)
accounts: (85, 7)
products: (7, 3)
sales_teams: (35, 3)
lead_sources: (8800, 7)


In [5]:
print(sp.dtypes)
print()
print(sp.isnull().sum())

opportunity_id        str
sales_agent           str
product               str
account               str
deal_stage            str
engage_date           str
close_date            str
close_value       float64
dtype: object

opportunity_id       0
sales_agent          0
product              0
account           1425
deal_stage           0
engage_date        500
close_date        2089
close_value       2089
dtype: int64


In [6]:
print(leads.dtypes)
print()
print(leads.isnull().sum())

opportunity_id           str
lead_source              str
lead_score           float64
cost_per_lead_inr    float64
lead_created_date        str
mql_date                 str
sql_date                 str
dtype: object

opportunity_id         0
lead_source            0
lead_score             0
cost_per_lead_inr      0
lead_created_date    500
mql_date             500
sql_date             500
dtype: int64


In [7]:
for c in ['engage_date', 'close_date']:
    sp[c] = pd.to_datetime(sp[c])

for c in ['lead_created_date', 'mql_date', 'sql_date']:
    leads[c] = pd.to_datetime(leads[c])

In [8]:
print(sp['close_value'].dtype)
print(leads['lead_score'].dtype)
print(leads['cost_per_lead_inr'].dtype)

float64
float64
float64


In [9]:
for c in ['sales_agent', 'product', 'account', 'deal_stage']:
    sp[c] = sp[c].astype('string').str.strip()

for c in ['sales_agent', 'manager', 'regional_office']:
    teams[c] = teams[c].astype('string').str.strip()

acc['account'] = acc['account'].astype('string').str.strip()
prod['product'] = prod['product'].astype('string').str.strip()

In [10]:
acc['sector'] = acc['sector'].astype('string').str.strip().str.lower()

sector_typo_fix = {
    'technolgy': 'technology'
}
acc['sector'] = acc['sector'].replace(sector_typo_fix)

In [11]:
bad_dates = sp[sp['close_date'] < sp['engage_date']]
print('Rows with close_date < engage_date:', len(bad_dates))

Rows with close_date < engage_date: 0


In [12]:
won_missing_value = sp[(sp['deal_stage'] == 'Won') & (sp['close_value'].isna())]
print('Won deals missing close_value:', len(won_missing_value))

Won deals missing close_value: 0


In [13]:
print(sp.groupby('deal_stage')['close_value'].apply(lambda s: s.notna().sum()))
print()
print(sp[sp['deal_stage'] == 'Lost']['close_value'].describe())

deal_stage
Engaging          0
Lost           2473
Prospecting       0
Won            4238
Name: close_value, dtype: int64

count    2473.0
mean        0.0
std         0.0
min         0.0
25%         0.0
50%         0.0
75%         0.0
max         0.0
Name: close_value, dtype: float64


In [14]:
orphan_agents = sp[~sp['sales_agent'].isin(teams['sales_agent'])]
orphan_products = sp[~sp['product'].isin(prod['product'])]
orphan_accounts = sp[sp['account'].notna() & ~sp['account'].isin(acc['account'])]
print('Orphan agents:', orphan_agents['sales_agent'].unique())
print('Orphan products:', orphan_products['product'].unique())
print('Orphan accounts:', orphan_accounts['account'].unique())

Orphan agents: <StringArray>
[]
Length: 0, dtype: string
Orphan products: <StringArray>
['GTXPro']
Length: 1, dtype: string
Orphan accounts: <StringArray>
[]
Length: 0, dtype: string


In [15]:
sp['product'] = sp['product'].replace({'GTXPro': 'GTX Pro'})

# re-check
orphan_products = sp[~sp['product'].isin(prod['product'])]
print('Orphan products after fix:', orphan_products['product'].unique())

Orphan products after fix: <StringArray>
[]
Length: 0, dtype: string


In [16]:
print(sp['opportunity_id'].duplicated().sum())

0


In [17]:
merge_check = sp[['opportunity_id']].merge(
    leads[['opportunity_id']], on='opportunity_id', how='left', indicator=True
)
print('Unmatched opportunity_ids:', (merge_check['_merge'] != 'both').sum())

dupe_leads = leads['opportunity_id'].duplicated().sum()
print('Duplicate opportunity_id in lead_sources:', dupe_leads)

Unmatched opportunity_ids: 0
Duplicate opportunity_id in lead_sources: 0


In [18]:
order_check = leads.dropna(subset=['lead_created_date', 'mql_date', 'sql_date'])
bad_order = order_check[
    ~((order_check['lead_created_date'] < order_check['mql_date']) &
      (order_check['mql_date'] < order_check['sql_date']))
]
print('Rows with an impossible date order:', len(bad_order))

Rows with an impossible date order: 0


In [19]:
print(sp.isnull().sum())
print()
print('Rows still in Prospecting/Engaging (no close_date/close_value expected):')
print(sp[sp['close_date'].isna()]['deal_stage'].value_counts())

opportunity_id       0
sales_agent          0
product              0
account           1425
deal_stage           0
engage_date        500
close_date        2089
close_value       2089
dtype: int64

Rows still in Prospecting/Engaging (no close_date/close_value expected):
deal_stage
Engaging       1589
Prospecting     500
Name: count, dtype: Int64


In [20]:
sp.to_csv('data/cleaned/sales_pipeline_cleaned.csv', index=False)
acc.to_csv('data/cleaned/accounts_cleaned.csv', index=False)
prod.to_csv('data/cleaned/products_cleaned.csv', index=False)
teams.to_csv('data/cleaned/sales_teams_cleaned.csv', index=False)
leads.to_csv('data/cleaned/lead_sources_cleaned.csv', index=False)


## Summary of cleaning decisions

| Issue | Decision |
|---|---|
| `account` nulls (1,425) | Kept — means "no linked account record," not an error |
| `engage_date` nulls (500) | Kept — deal still in Prospecting |
| `close_date`/`close_value` nulls (2,089) | Kept — deal still open; excluded from win-rate/revenue math downstream, not deleted |
| `close_value = 0` on Lost deals | Kept as-is — legitimate zero revenue, not missing data |
| `"technolgy"` sector typo | Corrected to `"technology"` |
| `"GTXPro"` vs `"GTX Pro"` product mismatch | Renamed to `"GTX Pro"` to match `products` table |
| Whitespace in text/key columns | Stripped across all tables before any join |
| Referential integrity (agents/products/accounts) | Verified — no orphans after the product fix |
| `lead_sources` join & date order | Verified — 1:1 join, no duplicates, chronological order holds |